# ECG Signal Analysis Using Python

## MIT-BIH Arrhythmia Database

This project analyzes real ECG signals from the MIT-BIH Arrhythmia Database using Python.

Main objectives:

- Load real ECG data
- Visualize ECG signals
- Detect R peaks
- Calculate RR intervals
- Estimate heart rate
- Compare detected peaks with expert annotations
- Evaluate detection performance

## 1. Background

Electrocardiography (ECG) records the electrical activity of the heart.

A typical ECG waveform contains:

- P wave: atrial depolarization
- QRS complex: ventricular depolarization
- T wave: ventricular repolarization

The R peak is commonly used to estimate heart rate and RR intervals.

## 2. Dataset

The ECG data used in this project comes from the MIT-BIH Arrhythmia Database on PhysioNet.

Record used:

- Record: 100
- Sampling frequency: 360 Hz
- ECG lead: first available lead
- Analysis duration: 60 seconds

## 3. Import Libraries

In [ ]:
import wfdb
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

## 4. Load ECG Data

In [ ]:
record = wfdb.rdrecord(
    "101",
    pn_dir="mitdb"
)

ecg = record.p_signal[:, 0]
fs = record.fs

print("Sampling frequency:", fs)
print("Signal length:", len(ecg))
print("Lead names:", record.sig_name)

In [ ]:
duration = 60

samples = int(fs * duration)

ecg_segment = ecg[:samples]

time = np.arange(samples) / fs

print("Duration:", duration, "seconds")
print("Number of samples:", samples)

## 5. ECG Visualization

The first 10 seconds of the ECG signal are displayed below.

In [ ]:
display_seconds = 10

display_samples = int(fs * display_seconds)

plt.figure(figsize=(14, 4))

plt.plot(
    time[:display_samples],
    ecg_segment[:display_samples]
)

plt.xlabel("Time (s)")
plt.ylabel("Amplitude (mV)")
plt.title("MIT-BIH Record 100 ECG")

plt.grid()
plt.show()

## 6. R Peak Detection

R peaks are detected using the `find_peaks()` function from SciPy.

A minimum height threshold and minimum distance between peaks are used to reduce false detections.

In [ ]:
peaks, properties = find_peaks(
    ecg_segment,
    height=0.5,
    distance=int(0.5 * fs)
)

print("Number of detected R peaks:", len(peaks))

In [ ]:
peaks_display = peaks[peaks < display_samples]

plt.figure(figsize=(14, 4))

plt.plot(
    time[:display_samples],
    ecg_segment[:display_samples],
    label="ECG"
)

plt.plot(
    time[peaks_display],
    ecg_segment[peaks_display],
    "x",
    markersize=8,
    label="Detected R Peaks"
)

plt.xlabel("Time (s)")
plt.ylabel("Amplitude (mV)")
plt.title("R Peak Detection")

plt.legend()
plt.grid()
plt.show()

## 7. Heart Rate Analysis

RR intervals are calculated from the time difference between consecutive R peaks.

Heart rate is calculated using:

Heart Rate = 60 / RR Interval

In [ ]:
rr_intervals = np.diff(peaks) / fs

heart_rates = 60 / rr_intervals

average_hr = np.mean(heart_rates)

print("Average heart rate:", round(average_hr, 2), "bpm")
print("Minimum heart rate:", round(np.min(heart_rates), 2), "bpm")
print("Maximum heart rate:", round(np.max(heart_rates), 2), "bpm")

In [ ]:
heart_rate_times = time[peaks[1:]]

plt.figure(figsize=(12, 4))

plt.plot(
    heart_rate_times,
    heart_rates,
    marker="o"
)

plt.xlabel("Time (s)")
plt.ylabel("Heart Rate (bpm)")
plt.title("Heart Rate Over Time")

plt.grid()
plt.show()

## 8. Expert Annotation Comparison

The detected R peaks are compared with the expert annotations provided by the MIT-BIH database.

In [9]:
annotation = wfdb.rdann(
    "100",
    "atr",
    pn_dir="mitdb"
)

expert_peaks = annotation.sample

expert_peaks_segment = expert_peaks[
    expert_peaks < samples
]

print(
    "Number of expert annotations:",
    len(expert_peaks_segment)
)

Number of expert annotations: 75


In [ ]:
expert_display = expert_peaks_segment[
    expert_peaks_segment < display_samples
]

plt.figure(figsize=(14, 4))

plt.plot(
    time[:display_samples],
    ecg_segment[:display_samples],
    label="ECG"
)

plt.plot(
    time[peaks_display],
    ecg_segment[peaks_display],
    "x",
    markersize=8,
    label="Our Detection"
)

plt.plot(
    time[expert_display],
    ecg_segment[expert_display],
    "o",
    fillstyle="none",
    markersize=10,
    label="Expert Annotation"
)

plt.xlabel("Time (s)")
plt.ylabel("Amplitude (mV)")
plt.title("Detected R Peaks vs Expert Annotation")

plt.legend()
plt.grid()
plt.show()

## 9. Performance Evaluation

Detected peaks are matched with expert annotations using a tolerance of 50 milliseconds.

Metrics used:

- True Positive (TP)
- False Positive (FP)
- False Negative (FN)
- Sensitivity
- Precision
- F1 Score

In [ ]:
tolerance = int(0.05 * fs)

tp = 0
matched_expert = set()

for p in peaks:

    differences = np.abs(
        expert_peaks_segment - p
    )

    closest_index = np.argmin(differences)

    if differences[closest_index] <= tolerance:

        if closest_index not in matched_expert:
            tp += 1
            matched_expert.add(closest_index)

fp = len(peaks) - tp

fn = (
    len(expert_peaks_segment)
    - len(matched_expert)
)

print("TP:", tp)
print("FP:", fp)
print("FN:", fn)

In [ ]:
sensitivity = tp / (tp + fn)

precision = tp / (tp + fp)

f1_score = (
    2
    * precision
    * sensitivity
    / (precision + sensitivity)
)

print(
    "Sensitivity:",
    round(sensitivity * 100, 2),
    "%"
)

print(
    "Precision:",
    round(precision * 100, 2),
    "%"
)

print(
    "F1 Score:",
    round(f1_score * 100, 2),
    "%"
)

## 10. Conclusion

In this project, a real ECG signal from the MIT-BIH Arrhythmia Database was analyzed using Python.

The project successfully implemented:

- ECG data loading
- ECG visualization
- R peak detection
- RR interval calculation
- Heart rate estimation
- Comparison with expert annotations
- Basic algorithm performance evaluation

This project demonstrates a basic biomedical signal processing workflow using Python.

### Limitations

The current analysis only uses one ECG record and a simple threshold-based R peak detection algorithm.

Future work could include:

- Testing more patients
- ECG filtering and noise removal
- Improved R peak detection algorithms
- Arrhythmia classification
- Machine learning